# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiezugwu/Data-Science/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
Rule: A declining page gets a high score if it was under 60 days old when the last 30 days began and had fewer than 50 impressions in its earliest 30 days, and it uses no outcome-window data and no product flags.

Reason code: NEW_PAGE_NO_HISTORY means the page is too young to have a stable track record, so its drop may come from an unsettled start instead of lost rankings.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Implement your scoring logic, rank the dataset, and output the result to CSV here.
# Once this cell executes successfully, the check cell below will load your file.

# Example template (uncomment and modify to match your logic):
import pandas as pd

# 1. Load your raw dataset
D = pd.read_parquet("decline_recovery.parquet")
#
# # 2. Implement your scoring logic and reason codes
# # 3. Sort / Rank the pages
import numpy as np

# label (used ONLY to evaluate, never to score)
D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100

# inputs: earlier windows and static facts only
D["impressions_first30"] = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["age_at_outcome_start_days"] = D.content_age_days - 30

# frozen thresholds
AGE_FLOOR, HISTORY_FLOOR = 60, 50
new_page   = (D.age_at_outcome_start_days < AGE_FLOOR).astype(int)
no_history = (D.impressions_first30 < HISTORY_FLOOR).astype(int)

# transparent score: one point per warning sign
D["score"] = new_page + no_history

# reason code and action label
D["reason_code"] = np.select(
    [(new_page == 1) & (no_history == 1), new_page == 1, no_history == 1],
    ["NEW_PAGE_NO_HISTORY", "NEW_PAGE", "NO_EARLY_HISTORY"],
    default="NONE")
D["action"] = np.where(D.score == 2, "Check first", np.where(D.score == 1, "Review soon", "Review later"))

# rank everything: higher score first; ties -> less early history, then more prior traffic
D = D.sort_values(["score", "impressions_first30", "impressions_prev_30d"],
                  ascending=[False, True, False]).reset_index(drop=True)
D["rank"] = D.index + 1

# evaluate precision@K against the base rate
severe = (D.severity >= D.severity.quantile(.75)).to_numpy()
def p_at_k(k): return severe[:k].mean()

# # 4. Save to the required path
os.makedirs('work/outputs', exist_ok=True)
D.to_csv('work/outputs/baseline_action_score.csv', index=False)

In [ ]:
import pandas as pd
df= pd.read_csv('work/outputs/baseline_action_score.csv')
df.head()

/tmp/ipykernel_2538/826730572.py:2: DtypeWarning: Columns (41) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv('work/outputs/baseline_action_score.csv')


,client_hash_id,content_hash_id,keyword_hash_id,keyword_char_count,keyword_token_count,public_url_hash_id,public_url_char_count,public_url_path_depth,content_title_hash_id,content_title_char_count,...,ai_opportunity,is_underperformer,is_declining,severity,impressions_first30,age_at_outcome_start_days,score,reason_code,action,rank
0,client_8ddc46da5414ffd8,content_9c458e21285ad07b,keyword_37285b8eba21d31d,38,8,url_0b4bccdcf604fece,90,3,title_3210a527ae33ef5d,38,...,NaN,False,True,81.001185,0,22,2,NEW_PAGE_NO_HISTORY,Check first,1
1,client_8ddc46da5414ffd8,content_8c82c1b4dfb5758f,keyword_1d12ed40de03ca2a,43,6,url_f49191097e3e4715,95,3,title_9ba1c44f85bae055,44,...,NaN,False,True,60.548626,0,30,2,NEW_PAGE_NO_HISTORY,Check first,2
2,client_8ddc46da5414ffd8,content_6cf9f71ac58292db,keyword_bea23779c83dc58e,35,6,url_5c3999a41ba4da0d,102,3,title_ee78f36c483a9512,51,...,NaN,False,True,73.815334,0,25,2,NEW_PAGE_NO_HISTORY,Check first,3
3,client_86ebc2f12c01f586,content_5e23ef5347a00a5e,keyword_a881e470096f4173,44,7,url_e133f423b9263d1c,101,3,title_61306a626f67877f,45,...,False,False,True,83.668904,0,28,2,NEW_PAGE_NO_HISTORY,Check first,4
4,client_8ddc46da5414ffd8,content_e910a28e4c886161,keyword_abd7c92ebe7d84df,30,5,url_bdbf8e9226e95d83,100,3,title_3e5ec928bd13b54e,48,...,NaN,False,True,84.271504,0,15,2,NEW_PAGE_NO_HISTORY,Check first,5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
Age (days)	Prior impressions / CTR	Confidence note	What would make it wrong	Actual drop
1 	22	50,640 / 2.8%	Low-med: the most traffic and a healthy CTR	A healthy page settling after launch, so nothing to fix	81.0%
2 	30	35,033 / 0.4%	Low-med: a lot of traffic to lose	A site-wide event at the client, or launch decay	60.5%
3 	25	25,408 / 0.5%	Low-med: a lot of traffic to lose	Same	73.8%
4	28	17,880 / 0.1%	Low: near-zero CTR means little real engagement	A spike that was never real traffic	83.7%
5 	15	17,090 / 0.2%	Low: the prior window is mostly launch	Launch decay or a client-wide event	84.3%
6	53	16,598 / 0.1%	Low: barely “new”, the weakest basis for the code	An indexing delay or a thin first window, not newness	63.0%
7 	32	15,346 / 0.1%	Low	A client-wide event	87.4%
8 	25	15,243 / 1.3%	Low-med: decent CTR	A healthy page settling	55.4%
9	51	12,490 / 0.1%	Low: barely “new”	An indexing delay, not newness	37.3%
10	27	12,227 / 0.1%	Low	Normal launch decay	48.1%
11 	7	12,114 / 0.3%	Low: only 7 days old, so the prior window is nearly all launch	Launch decay or a client-wide event	62.5%
12 	30	11,152 / 1.6%	Low–med: strong CTR	A healthy page settling	67.6%
13 	12	10,838 / 0.3%	Low: 12 days old	Launch decay or a client-wide event	51.2%
14 	22	9,872 / 0.3%	Low	Same	39.5%
15 	15	9,850 / 0.1%	Low: 15 days old, near-zero CTR	Launch decay	66.0%
16	24	9,346 / 0.5%	Low	Normal launch decay	41.1%
17 	22	9,041 / 0.1%	Low	A spike with no real engagement	95.1% ✓
18	32	8,893 / 0.1%	Low	Normal launch decay	64.0%
19	26	8,711 / 0.8%	Low–med	A healthy page settling	43.6%
20 	23	7,730 / 0.3%	Low	A client-wide event	72.7%

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weak in hindsight (dropped under 50%): ranks 9, 10, 14, 16 and 19.

Questionable from the inputs alone: ranks 1, 8 and 12 have a healthy prior CTR (1.3% to 2.8%). Ranks 6 and 9 are barely “new” at 51 to 53 days.

The 13 pages from one client aren't 13 independent picks. That client has 951 pages in the data, and 100% of them are new (under 60 days at the start of the outcome window). That points to a bulk launch, so the real question is client-level, not page-level.

Keyword articles are 98% of the population (55,233 of 56,253). It's the base rate, not a signal. By content type, the flagged share is 7.8% for keyword articles, 11% for feedly article (n=638) and 0% for comparison article (n=382), so the rule isn't a content-type prox

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.